In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ayomidefagbolade/combined-df/combined_data.csv


In [2]:
#!pip install vllm

In [3]:
#!pip install "protobuf<6.0.0" --force-reinstall

In [4]:
# Delete specific checkpoint/log files
!rm -f label_vocab_checkpoint.json progress_checkpoint.json label_vocab_final.json failed_batches.log

# OR: Clear EVERYTHING inside /kaggle/working directory
!rm -rf /kaggle/working/*

In [5]:
import pandas as pd

# Read the CSV file into df
combined_df = pd.read_csv('/kaggle/input/datasets/ayomidefagbolade/combined-df/combined_data.csv')

# Display the first few rows
combined_df .head()

,combined_text
0,sorghum low local production of grains
1,banana/plantain can be used as a remedy for co...
2,"sorghum prices decreased in many regions, with..."
3,banana/plantain Banana Bacteria Wilt disease i...
4,cassava funding for production


In [6]:
import random
# 1. Clear out previous zombie processes from your crash
!pkill -f vllm

# 2. FORCE vLLM back to the stable V0 architecture (Fixes Kaggle Multi-GPU)
os.environ["VLLM_VERSION"] = "v0"

# 3. Apply standard multi-GPU network overrides
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["VLLM_HOST_IP"] = "127.0.0.1"
os.environ["GLOO_SOCKET_IFNAME"] = "lo"
os.environ["NCCL_SOCKET_IFNAME"] = "lo"
os.environ["MASTER_PORT"] = str(random.randint(20000, 30000))

In [7]:

import json
import os
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams

# ...

# ── 1. Load model ────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct-AWQ"

llm = LLM(
    model=MODEL_NAME,
    dtype="auto",
    max_model_len=20000,          # raised: vocab block is now echoed in BOTH input and output each batch
    gpu_memory_utilization=0.75,
    disable_custom_all_reduce=True,
    tensor_parallel_size=2,
)

INFO 07-22 01:19:33 [api_utils.py:273] non-default args: {'max_model_len': 20000, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'disable_custom_all_reduce': True, 'model': 'Qwen/Qwen2.5-14B-Instruct-AWQ'}
WARNING 07-22 01:19:33 [envs.py:2041] Unknown vLLM environment variable detected: VLLM_VERSION


INFO 07-22 01:19:33 [model.py:619] Resolved architecture: Qwen2ForCausalLM
INFO 07-22 01:19:33 [model.py:1776] Using max model len 20000
INFO 07-22 01:19:34 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files:   0%|          | 0/3 [00:00<?, ?it/s]

INFO 07-22 01:19:35 [vllm.py:1042] Asynchronous scheduling is enabled.
INFO 07-22 01:19:35 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=11774) INFO 07-22 01:19:39 [core.py:114] Initializing a V1 LLM engine (v0.25.1) with config: model='Qwen/Qwen2.5-14B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-14B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=20000, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=True, quantization=auto_awq, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backe

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(Worker_TP0 pid=11792) INFO 07-22 01:19:48 [default_loader.py:430] Loading weights took 2.73 seconds
(Worker_TP0 pid=11792) INFO 07-22 01:19:51 [model_runner.py:302] Model loading took 4.67 GiB and 7.357102 seconds
(Worker_TP0 pid=11792) WARNING 07-22 01:19:51 [topk_topp_sampler.py:62] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(Worker_TP1 pid=11793) INFO 07-22 01:19:52 [model_runner.py:302] Model loading took 4.67 GiB and 8.645996 seconds
(Worker_TP0 pid=11792) INFO 07-22 01:20:04 [backends.py:1089] Using cache directory: /root/.cache/vllm/torch_compile_cache/615d5060df/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=11792) INFO 07-22 01:20:04 [backends.py:1148] Dynamo bytecode transform time: 10.60 s
(Worker_TP1 pid=11793) INFO 07-22 01:20:10 [decorators.py:311] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/323e2a35a51b0c57130b

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:08<00:00,  4.29it/s]


(Worker_TP0 pid=11792) INFO 07-22 01:20:42 [model_runner.py:722] Graph capturing finished in 23 secs, took 1.75 GiB
(Worker_TP1 pid=11793) INFO 07-22 01:20:42 [model_runner.py:722] Graph capturing finished in 23 secs, took 1.75 GiB
(Worker_TP0 pid=11792) (Worker_TP1 pid=11793) INFO 07-22 01:20:43 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
INFO 07-22 01:20:43 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=11774) INFO 07-22 01:20:44 [core.py:337] init engine (profile, create kv cache, warmup model) took 51.22 s (compilation: 17.10 s)
(EngineCore pid=11774) INFO 07-22 01:20:44 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(Worker_TP0 pid=11792) WARNING 07-22 01:20:46 [jit_monitor.py:129] Triton kernel JIT compilation during inference: kernel_un

In [8]:
# ── 2. JSON schema for guided decoding — forces valid structured output ──
LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "labels": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "label": {"type": "string"},
                    "definition": {"type": "string"},
                },
                "required": ["label", "definition"],
            },
        }
    },
    "required": ["labels"],
}

structured_params = StructuredOutputsParams(json=LABEL_SCHEMA)

sampling_params = SamplingParams(
    temperature=0.01,
    structured_outputs=structured_params,
    max_tokens=16000,  # raised: model now outputs the FULL vocab each batch, not just new additions
)

In [9]:
SYSTEM_INSTRUCTION_TEMPLATE = """You are an expert taxonomy curator extracting structured topic labels from short agricultural statements (grant descriptions and news excerpts) regarding cassava, sweet potato, yam, sorghum, and banana/plantain.

Your task is to read the statements below and extract SPECIFIC, DISTINCT topic labels that describe what each statement (or group of similar statements) is about. You are not maintaining a list across batches — just extract what's genuinely present in THIS batch.

EXTRACTION RULES:
1. BE SPECIFIC, NOT GENERIC. Prefer narrow, concrete labels over broad umbrella terms.
   - Bad: "food security", "agriculture", "development", "research and development"
   - Good: "post-harvest loss", "seed certification", "drought-tolerant varieties", "extension worker training"

2. Do NOT use crop names (e.g. "cassava", "sweet potato") as labels or part of label names.

3. Do NOT create overly narrow, single-event labels (e.g. "july 2023 flood in kampala").

4. If multiple statements in this batch describe the same concept, extract it ONCE as a single label, not repeated.

5. LABEL FORMAT: "label" — short, lowercase noun phrase (2-4 words). "definition" — exactly one clear, objective sentence defining the scope of the label and how it applies across crops.

OUTPUT INSTRUCTIONS:
- Respond strictly using the required JSON schema. No surrounding explanation, markdown, or commentary.
"""

In [10]:
def build_vocab_block(vocab):
    if not vocab:
        return "(empty — no categories defined yet)"
    lines = []
    for cat, info in vocab.items():
        lines.append(f"## {cat}: {info['definition']}")
        for label, definition in info["subtopics"].items():
            lines.append(f"  - {label}: {definition}")
    return "\n".join(lines)

def build_prompt(rows):
    statements = "\n".join(f"- {text}" for text in rows)
    return (
        f"<|im_start|>system\n{SYSTEM_INSTRUCTION_TEMPLATE}<|im_end|>\n"
        f"<|im_start|>user\n{statements}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

In [11]:
CHECKPOINT_PROGRESS_PATH = "progress_checkpoint.json"
OUTPUT_CSV_PATH = "extracted_labels.csv"

def save_progress(next_start_index):
    tmp = CHECKPOINT_PROGRESS_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump({"next_start_index": next_start_index}, f)
    os.replace(tmp, CHECKPOINT_PROGRESS_PATH)

def load_progress():
    if os.path.exists(CHECKPOINT_PROGRESS_PATH):
        with open(CHECKPOINT_PROGRESS_PATH) as f:
            return json.load(f)["next_start_index"]
    return 0

def append_labels_to_csv(batch_start, labels):
    import csv
    write_header = not os.path.exists(OUTPUT_CSV_PATH)
    with open(OUTPUT_CSV_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(["batch_start", "label", "definition"])
        for item in labels:
            writer.writerow([batch_start, item["label"].strip().lower(), item["definition"]])

In [12]:
def load_checkpoint_if_exists():
    vocab = {
        "crop health": {
            "definition": "Topics related to biological threats and health status of the crop, including diseases and pests.",
            "subtopics": {
                "crop disease": "Mentions of plant diseases, pathogens, or disease outbreaks affecting the crop.",
                "crop pest": "Mentions of insect pests, rodents, or other pest damage affecting the crop.",
            },
        },
        "farm inputs": {
            "definition": "Physical resources and materials provided to farmers to support production.",
            "subtopics": {
                "farm input": "Agricultural inputs such as fertilizer, tools, or equipment provided to farmers (excludes seeds/cuttings, see 'planting material').",
                "planting material": "Seeds, cuttings, vines, or vegetative propagation material (e.g. cassava stems, sweet potato vines) distributed or developed for planting.",
            },
        },
    }
    resume_from = 0
    if os.path.exists(CHECKPOINT_VOCAB_PATH):
        with open(CHECKPOINT_VOCAB_PATH) as f:
            vocab = json.load(f)
    if os.path.exists(CHECKPOINT_PROGRESS_PATH):
        with open(CHECKPOINT_PROGRESS_PATH) as f:
            resume_from = json.load(f)["next_start_index"]
    return vocab, resume_from

In [13]:
BATCH_SIZE = 500

texts = combined_df["combined_text"].tolist()
resume_from = load_progress()

batch_starts = [s for s in range(0, len(texts), BATCH_SIZE) if s >= resume_from]

for start in batch_starts:
    batch_texts = texts[start:start + BATCH_SIZE]
    prompt = build_prompt(batch_texts)

    try:
        outputs = llm.generate([prompt], sampling_params)
        raw = outputs[0].outputs[0].text.strip()
        parsed = json.loads(raw)

        if not isinstance(parsed, dict) or "labels" not in parsed:
            raise ValueError(f"Unexpected shape: {parsed}")

        append_labels_to_csv(start, parsed["labels"])
        print(f"Batch {start}-{start+BATCH_SIZE} done. Extracted {len(parsed['labels'])} labels.")

    except Exception as e:
        print(f"Batch {start} failed: {e}")
        with open("failed_batches.log", "a") as f:
            f.write(f"batch start {start}: {e}\n")
            f.write(f"raw output (tail): {raw[-1000:]!r}\n\n")

    save_progress(start + BATCH_SIZE)

print("Extraction pass complete. Raw labels written to", OUTPUT_CSV_PATH)

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [02:17<00:00, 137.47s/it, est. speed input: 70.61 toks/s, output: 12.81 toks/s]

Batch 0-500 done. Extracted 41 labels.


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [12:13<00:00, 733.51s/it, est. speed input: 12.92 toks/s, output: 14.34 toks/s]

...

Processed prompts: 100%|██████████| 1/1 [01:15<00:00, 75.33s/it, est. speed input: 121.69 toks/s, output: 10.17 toks/s]

Batch 18000-18500 done. Extracted 22 labels.


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [01:52<00:00, 112.27s/it, est. speed input: 75.95 toks/s, output: 14.06 toks/s]

Batch 18500-19000 done. Extracted 41 labels.
Extraction pass complete. Raw labels written to extracted_labels.csv


In [14]:
label_vocab

NameError: name 'label_vocab' is not defined